# Panerai Production ETL Pipeline
This notebook implements the production-ready, incremental ETL pipeline using the Medallion Architecture.

## Pipeline Overview
1. **Bronze**: Incremental ingestion via Auto Loader (`cloudFiles`).
2. **Silver**: Incremental `MERGE` (Upsert) with URL validation and deduplication.
3. **Gold**: Star Schema generation with Dimension and Fact tables.

In [ ]:
import os
from pyspark.sql import functions as F
from pyspark.sql import Window
import yaml

# --- Production Configuration ---
# Use dbutils.widgets for parameterization in Databricks Jobs
try:
    env = dbutils.widgets.get("env")
except:
    env = "dev"

BRONZE_TABLE = f"panerai_project.{env}.panerai_data_bronze_{env}"
SILVER_TABLE = f"panerai_project.{env}.panerai_data_silver_{env}"
VOLUME_PATH = f"/Volumes/panerai_project/{env}/panerai_data"
CHECKPOINT_PATH = f"/Volumes/panerai_project/{env}/_checkpoints/bronze_ingestion"

EXCHANGE_RATES = {
    "USD": 0.88, "JPY": 0.0056, "AED": 0.24, 
    "CHF": 1.06, "GBP": 1.16, "EUR": 1.0
}

## 1. Bronze Layer: Auto Loader Ingestion
We use `cloudFiles` to only process new files arriving in the volume since the last run.

In [ ]:
# Auto Loader stream
bronze_stream = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .option("recursiveFileLookup", "true")
    .load(VOLUME_PATH))

# Add metadata and write to Bronze Delta Table
query_bronze = (bronze_stream
    .withColumn("file_path", "_metadata.file_path")
    .withColumn("ingested_at", "_metadata.current_timestamp")
    .writeStream
    .format("delta")
    .option("checkpointLocation", CHECKPOINT_PATH)
    .option("mergeSchema", "true")
    .trigger(availableNow=True) # Process all new data then stop
    .toTable(BRONZE_TABLE))

print(f"Started incremental ingestion to {BRONZE_TABLE}...")

## 2. Silver Layer: Incremental Upsert
We process only the new data from Bronze and MERGE it into Silver.

In [ ]:
def process_silver_batch(df):
    # --- Load YAML Corrections ---
    try:
        with open("./ref_corrected_url.yaml", "r") as f:
            ref_corrected_url = yaml.safe_load(f)
    except:
        ref_corrected_url = {}

    # 1. Basic Cleaning
    df = df.dropna(subset=["reference"])
    
    # 2. URL Correction
    if ref_corrected_url:
        corrections = [(ref, country, data["url"]) for ref, countries in ref_corrected_url.items() for country, data in countries.items()]
        corr_df = spark.createDataFrame(corrections, ["reference", "country", "correct_url"])
        df = df.join(F.broadcast(corr_df), on=["reference", "country"], how="left")
        df = df.withColumn("url", F.coalesce(F.col("correct_url"), F.col("url"))).drop("correct_url")
    
    # 3. Normalization
    df = df.withColumn("price_clean", F.regexp_replace(F.col("price"), r"[^0-9]", "").cast("int"))
    df = df.withColumn("diameter_mm", F.regexp_replace(F.col("diameter_tag"), r"[^0-9]", "").cast("int"))
    
    rate_expr = F.lit(1.0)
    for curr, rate in EXCHANGE_RATES.items():
        rate_expr = F.when(F.col("currency") == curr, F.lit(rate)).otherwise(rate_expr)
    df = df.withColumn("price_eur", F.col("price_clean") * rate_expr)
    
    # 4. Imputation
    df = df.withColumn("edition", F.coalesce(F.col("edition"), F.lit("unknown")))
    df = df.withColumn("comment", F.coalesce(F.col("comment"), F.lit("unknown")))
    
    # 5. Deduplication (Keep latest record per ref/country/date)
    window_spec = Window.partitionBy("reference", "country", "date").orderBy(F.col("ingested_at").desc())
    df = df.withColumn("rn", F.row_number().over(window_spec)).filter("rn = 1").drop("rn")
    
    return df

# Fetch new data since last merge (simulated by reading Bronze)
silver_updates = process_silver_batch(spark.read.table(BRONZE_TABLE))
silver_updates.createOrReplaceTempView("silver_updates_view")

spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")

spark.sql(f"""
MERGE INTO {SILVER_TABLE} AS target
USING silver_updates_view AS source
ON target.reference = source.reference 
   AND target.country = source.country 
   AND target.date = source.date
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")

print(f"✅ Incremental merge to {SILVER_TABLE} complete.")

## 3. Gold Layer: Star Schema
We rebuild the gold layer based on the updated silver data.

In [ ]:
silver_final = spark.read.table(SILVER_TABLE)

# Dim Collection
dim_col = silver_final.select("collection").distinct().withColumn("collection_id", F.monotonically_increasing_id())
dim_col.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"panerai_project.{env}.dim_collection")

# Dim Country
dim_cnt = silver_final.select("country", "currency").distinct().withColumn("country_id", F.monotonically_increasing_id())
dim_cnt.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"panerai_project.{env}.dim_country")

# Dim Watch
dim_w = silver_final.select("reference", "title", "edition", "comment").distinct()
dim_w.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"panerai_project.{env}.dim_watch")

# Fact Pricing
fact_p = silver_final.join(dim_col, "collection").join(dim_cnt, "country") \
               .select("reference", "date", "price_eur", "diameter_mm", "collection_id", "country_id")
fact_p.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"panerai_project.{env}.fact_pricing")

print("✅ Gold layer tables updated.")